In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

engine = pd.read_csv("../../data/processed/engine_cleaned.csv")
print(engine.shape)
print(engine.head())
print("\nRUL range:", engine["rul"].min(), "to", engine["rul"].max())

(20631, 22)
   unit_id  cycle  op_setting_1  op_setting_2  sensor_2  sensor_3  sensor_4  \
0        1      1       -0.0007       -0.0004    641.82   1589.70   1400.60   
1        1      2        0.0019       -0.0003    642.15   1591.82   1403.14   
2        1      3       -0.0043        0.0003    642.35   1587.99   1404.20   
3        1      4        0.0007        0.0000    642.35   1582.79   1401.87   
4        1      5       -0.0019       -0.0002    642.37   1582.85   1406.22   

   sensor_5  sensor_6  sensor_7  ...  sensor_11  sensor_12  sensor_13  \
0     14.62     21.61    554.36  ...      47.47     521.66    2388.02   
1     14.62     21.61    553.75  ...      47.49     522.28    2388.07   
2     14.62     21.61    554.26  ...      47.27     522.42    2388.03   
3     14.62     21.61    554.45  ...      47.13     522.86    2388.08   
4     14.62     21.61    554.00  ...      47.28     522.19    2388.04   

   sensor_14  sensor_15  sensor_16  sensor_17  sensor_20  sensor_21  rul  

In [2]:
fig = px.histogram(engine, x="rul", nbins=50,
                   title="Aircraft Engine — RUL Distribution",
                   labels={"rul": "Remaining Useful Life (cycles)"},
                   color_discrete_sequence=["#2563eb"])
fig.show()

print("Mean RUL:", round(engine["rul"].mean(), 2))
print("Median RUL:", engine["rul"].median())

Mean RUL: 107.81
Median RUL: 103.0


In [3]:
unit1 = engine[engine["unit_id"] == 1].copy()

sensors_to_plot = ["sensor_2", "sensor_7", "sensor_11", "sensor_12"]

for sensor in sensors_to_plot:
    fig = px.line(unit1, x="cycle", y=sensor,
                  title=f"Engine Unit 1 — {sensor} over cycles",
                  labels={"cycle": "Cycle", sensor: sensor})
    fig.show()

In [4]:
engine_numeric = engine.drop(columns=["unit_id"])

corr_with_rul = engine_numeric.corr()["rul"].drop("rul").sort_values()

fig = px.bar(x=corr_with_rul.index, y=corr_with_rul.values,
             title="Aircraft Engine — Sensor Correlation with RUL",
             labels={"x": "Sensor", "y": "Correlation"},
             color=corr_with_rul.values,
             color_continuous_scale="RdBu")
fig.show()

print("\nTop 3 positively correlated with RUL:\n", corr_with_rul.tail(3))
print("\nTop 3 negatively correlated with RUL:\n", corr_with_rul.head(3))


Top 3 positively correlated with RUL:
 sensor_12    0.671983
sensor_5          NaN
sensor_16         NaN
Name: rul, dtype: float64

Top 3 negatively correlated with RUL:
 cycle       -0.736241
sensor_11   -0.696228
sensor_4    -0.678948
Name: rul, dtype: float64


In [5]:
sample_units = engine["unit_id"].unique()[:5]
sample_df = engine[engine["unit_id"].isin(sample_units)]

fig = px.line(sample_df, x="cycle", y="sensor_11",
              color="unit_id",
              title="Aircraft Engine — Sensor 11 Degradation (5 engines)",
              labels={"cycle": "Cycle", "sensor_11": "Sensor 11", 
                      "unit_id": "Engine Unit"})
fig.show()

In [6]:
# Engine is considered "at risk" if RUL is 30 cycles or less
engine["failure_soon"] = (engine["rul"] <= 30).astype(int)

print("At risk distribution:")
print(engine["failure_soon"].value_counts())
print("At risk rate:", round(engine["failure_soon"].mean() * 100, 2), "%")

At risk distribution:
failure_soon
0    17531
1     3100
Name: count, dtype: int64
At risk rate: 15.03 %


In [7]:
for sensor in ["sensor_2", "sensor_7", "sensor_11", "sensor_12"]:
    engine[f"{sensor}_rolling"] = engine.groupby("unit_id")[sensor].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )

print("Rolling mean columns added.")
print(engine.shape)

Rolling mean columns added.
(20631, 27)


In [8]:
import os
os.makedirs("../../reports/figures", exist_ok=True)

engine.to_csv("../../data/processed/engine_cleaned.csv", index=False)
print("Saved updated engine CSV with new features.")

Saved updated engine CSV with new features.


In [9]:
# Drop near-zero variance sensors that returned NaN correlation
engine = engine.drop(columns=["sensor_5", "sensor_16"])
engine.to_csv("../../data/processed/engine_cleaned.csv", index=False)
print("Dropped sensor_5 and sensor_16. New shape:", engine.shape)

Dropped sensor_5 and sensor_16. New shape: (20631, 25)
